In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
    .appName("OtimizacaoJoin") \
    .getOrCreate()

In [4]:
df_video = spark.read.parquet("/content/videos-preparados.snappy (1).parquet")

In [5]:
df_video.show()

+--------------------+-----------+------------+----------------+------+--------+---------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+
|               Title|   Video ID|Published At|         Keyword| Likes|Comments|    Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|
+--------------------+-----------+------------+----------------+------+--------+---------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+
|ASMR MUKBANG DOUB...|--ZI0dSbbNU|  2020-04-18|         mukbang|378858|   18860| 17975269|   18372987|2020|    4|         30.0|[0.6985786560867407]|[0.02303716158264...|[378858.0,1.79752...|
|Deadly car bomb d...|--hxd1CrOqg|  2022-08-22|            news|  6379|    4853|   808787|     820019|2022|    8|         37.0|[0.8936407990235931]|[3.87946679100418...|[6379.0,808787.0,...|
|How Biden&#39;s s...|--ixiTypG8g|  2022-08-2

In [6]:
df_video.printSchema()

root
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: integer (nullable = true)
 |-- Interaction: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Keyword Index: double (nullable = true)
 |-- Features PCA: vector (nullable = true)
 |-- Features Normal: vector (nullable = true)
 |-- Features: vector (nullable = true)



In [7]:
df_comments = spark.read.parquet("/content/videos-comments-tratados.snappy (1).parquet")

In [8]:
df_comments.show()

+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+
|   Video ID|               Title|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|             Comment|Sentiment|Likes Comment|
+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Let's not forget ...|        1|           95|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Here in NZ 50% of...|        0|           19|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|I will forever ac...|        2|          161|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Whenever I go to ...|        0|            8|
|wAZZ-UWGVHI|

In [9]:
df_comments.printSchema()

root
 |-- Video ID: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: integer (nullable = true)
 |-- Interaction: integer (nullable = true)
 |-- Year: string (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Sentiment: integer (nullable = true)
 |-- Likes Comment: integer (nullable = true)



In [13]:
df_video.createOrReplaceTempView("videos")

In [14]:
df_comments.createOrReplaceTempView("comments")

In [16]:
join_video_comments = spark.sql("""

SELECT
    *
FROM videos v
INNER JOIN comments c
ON v.`Video ID` = c.`Video ID`

""")

In [17]:
join_video_comments.show()

+--------------------+-----------+------------+-------+-----+--------+-------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+
|               Title|   Video ID|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|   Video ID|               Title|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|             Comment|Sentiment|Likes Comment|
+--------------------+-----------+------------+-------+-----+--------+-------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+
|Apple Pay Is Kill...|wAZZ-UWGVHI|  2022-08-23

In [18]:
join_video_comments.show()

+--------------------+-----------+------------+-------+-----+--------+-------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+
|               Title|   Video ID|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|   Video ID|               Title|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|             Comment|Sentiment|Likes Comment|
+--------------------+-----------+------------+-------+-----+--------+-------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+
|Apple Pay Is Kill...|wAZZ-UWGVHI|  2022-08-23

In [19]:

# Reparticionando os dados dos vídeos em 4 partitions
# para melhorar o paralelismo durante o processamento
df_video_rep = df_video.repartition(4)

In [20]:
# Reparticionando os dados dos comentários
df_comments_rep = df_comments.repartition(4)

In [21]:
# Criando views temporárias dos dataframes reparticionados
df_video_rep.createOrReplaceTempView("videos_rep")

In [22]:

df_comments_rep.createOrReplaceTempView("comments_rep")

In [23]:
# Realizando INNER JOIN entre os dataframes reparticionados
join_rep = spark.sql("""

SELECT
    *
FROM videos_rep v
INNER JOIN comments_rep c
ON v.`Video ID` = c.`Video ID`

""")

In [24]:
join_rep.show()

+--------------------+-----------+------------+----------------+-------+--------+--------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+----------------+-------+--------+--------+-----------+----+--------------------+---------+-------------+
|               Title|   Video ID|Published At|         Keyword|  Likes|Comments|   Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|   Video ID|               Title|Published At|         Keyword|  Likes|Comments|   Views|Interaction|Year|             Comment|Sentiment|Likes Comment|
+--------------------+-----------+------------+----------------+-------+--------+--------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+----------------+-------+--------+--------+-----------+----+--------------------

In [25]:
join_rep.explain(True)

== Parsed Logical Plan ==
'Project [*]
+- 'Join Inner, ('v.Video ID = 'c.Video ID)
   :- 'SubqueryAlias v
   :  +- 'UnresolvedRelation [videos_rep], [], false
   +- 'SubqueryAlias c
      +- 'UnresolvedRelation [comments_rep], [], false

== Analyzed Logical Plan ==
Title: string, Video ID: string, Published At: date, Keyword: string, Likes: int, Comments: int, Views: int, Interaction: int, Year: int, Month: int, Keyword Index: double, Features PCA: vector, Features Normal: vector, Features: vector, Video ID: string, Title: string, Published At: date, Keyword: string, Likes: int, Comments: int, Views: int, Interaction: int, Year: string, Comment: string, Sentiment: int, ... 1 more fields
Project [Title#0, Video ID#1, Published At#2, Keyword#3, Likes#4, Comments#5, Views#6, Interaction#7, Year#8, Month#9, Keyword Index#10, Features PCA#11, Features Normal#12, Features#13, Video ID#58, Title#59, Published At#60, Keyword#61, Likes#62, Comments#63, Views#64, Interaction#65, Year#66, Comment

In [26]:
# Reduzindo a quantidade de partitions para diminuir custo de processamento
df_video_coalesce = df_video.coalesce(2)

In [27]:
df_comments_coalesce = df_comments.coalesce(2)

In [28]:
# Criando views temporárias dos dataframes com coalesce
df_video_coalesce.createOrReplaceTempView("videos_coalesce")

In [29]:
df_comments_coalesce.createOrReplaceTempView("comments_coalesce")

In [30]:
# Realizando INNER JOIN utilizando os dataframes com coalesce
join_coalesce = spark.sql("""

SELECT
    *
FROM videos_coalesce v
INNER JOIN comments_coalesce c
ON v.`Video ID` = c.`Video ID`

""")

In [31]:
join_coalesce.show()

+--------------------+-----------+------------+-------+-----+--------+-------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+
|               Title|   Video ID|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|   Video ID|               Title|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|             Comment|Sentiment|Likes Comment|
+--------------------+-----------+------------+-------+-----+--------+-------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+
|Apple Pay Is Kill...|wAZZ-UWGVHI|  2022-08-23

In [32]:
# Selecionando apenas colunas necessárias para reduzir uso de memória
videos_otimizado = df_video.select(
    "Video ID",
    "Title",
    "Views"
)

In [33]:
# Selecionando apenas colunas importantes do dataframe de comentários
comments_otimizado = df_comments.select(
    "Video ID",
    "Comment",
    "Sentiment"
)

In [34]:
# Filtrando apenas vídeos com mais de 1000 visualizações
# para reduzir o volume de dados processados
videos_filtrados = videos_otimizado.filter(
    videos_otimizado.Views > 1000
)

In [35]:
# Reparticionando os dataframes utilizando a chave do JOIN
# para melhorar a performance da junção
videos_filtrados = videos_filtrados.repartition(4, "Video ID")

In [36]:
comments_otimizado = comments_otimizado.repartition(4, "Video ID")

In [37]:
# Criando views temporárias otimizadas
videos_filtrados.createOrReplaceTempView("videos_filtrados")

In [38]:
comments_otimizado.createOrReplaceTempView("comments_otimizado")

In [39]:
join_otimizado = spark.sql("""

SELECT
    v.`Video ID`,
    v.Title,
    v.Views,
    c.Comment,
    c.Sentiment
FROM videos_filtrados v
INNER JOIN comments_otimizado c
ON v.`Video ID` = c.`Video ID`

""")

In [40]:
join_otimizado.show()

+-----------+--------------------+-------+--------------------+---------+
|   Video ID|               Title|  Views|             Comment|Sentiment|
+-----------+--------------------+-------+--------------------+---------+
|ErMwWXQxHp0|Best Back to Scho...|1855644|Guys, a quick not...|        1|
|ErMwWXQxHp0|Best Back to Scho...|1855644|"this is hilariou...|        1|
|ErMwWXQxHp0|Best Back to Scho...|1855644|Everyone has been...|        2|
|ErMwWXQxHp0|Best Back to Scho...|1855644|"Shoutout to my O...|        2|
|ErMwWXQxHp0|Best Back to Scho...|1855644|BEST BUY: We want...|     NULL|
|ErMwWXQxHp0|Best Back to Scho...|1855644|Most of the items...|        1|
|ErMwWXQxHp0|Best Back to Scho...|1855644|You don't necessa...|        2|
|ErMwWXQxHp0|Best Back to Scho...|1855644|Personally I feel...|        2|
|ErMwWXQxHp0|Best Back to Scho...|1855644|Hey everyone. Goo...|     NULL|
|ErMwWXQxHp0|Best Back to Scho...|1855644|I have used Galax...|        2|
|wLlL46pYcg4|15 Emerging Techn...|7001

In [41]:
join_otimizado.explain(True)

== Parsed Logical Plan ==
'Project ['v.Video ID, 'v.Title, 'v.Views, 'c.Comment, 'c.Sentiment]
+- 'Join Inner, ('v.Video ID = 'c.Video ID)
   :- 'SubqueryAlias v
   :  +- 'UnresolvedRelation [videos_filtrados], [], false
   +- 'SubqueryAlias c
      +- 'UnresolvedRelation [comments_otimizado], [], false

== Analyzed Logical Plan ==
Video ID: string, Title: string, Views: int, Comment: string, Sentiment: int
Project [Video ID#1, Title#0, Views#6, Comment#67, Sentiment#68]
+- Join Inner, (Video ID#1 = Video ID#58)
   :- SubqueryAlias v
   :  +- SubqueryAlias videos_filtrados
   :     +- View (`videos_filtrados`, [Video ID#1, Title#0, Views#6])
   :        +- RepartitionByExpression [Video ID#1], 4
   :           +- Filter (Views#6 > 1000)
   :              +- Project [Video ID#1, Title#0, Views#6]
   :                 +- Relation [Title#0,Video ID#1,Published At#2,Keyword#3,Likes#4,Comments#5,Views#6,Interaction#7,Year#8,Month#9,Keyword Index#10,Features PCA#11,Features Normal#12,Feature

In [42]:
join_otimizado.write.mode("overwrite").parquet(
    "join-videos-comments-otimizado"
)

In [43]:
spark.stop()